**Dataset**
labeled dataset collected from twitter (Hate Speech.tsv)

**Objective**
classify tweets containing hate speech from other tweets. <br>
0 -> no hate speech <br>
1 -> contains hate speech <br>

**Evaluation metric**
macro f1 score

**Steps**

To classify hate speech in tweets, follow these key steps:

1. **Data Preprocessing**: Clean text (remove punctuation, stopwords, etc.), lowercase, tokenize, and so on.
2. **Text Representation**: Use Bag of Words, TF-IDF, or word embeddings (e.g., GloVe, Word2Vec, or FastText).
3. **Modeling Approaches**:
   - **Traditional Models**: Logistic Regression, Naive Bayes, SVM, Random Forest.
   - **Deep Learning**: LSTM or RNN.
4. **Evaluation**
5. **Optimization**: Use hyperparameter tuning, regularization, and ensemble methods for better performance.


### Import used libraries

In [33]:
!pip install vaderSentiment
!pip install tensorflow
!pip install scikit-learn
!pip install pandas
!pip install scikeras

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import GridSearchCV
from gensim.models import Word2Vec
from sklearn.ensemble import VotingClassifier
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from scikeras.wrappers import KerasClassifier
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('vader_lexicon')

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Load Dataset

###### Note: search how to load the data from tsv file

In [3]:
file_path = "Hate Speech.tsv"
if not os.path.isfile(file_path):
    raise FileNotFoundError(f"The file '{file_path}' was not found. Please check the file path.")
try:
    df = pd.read_csv(file_path, sep="\t")
except Exception as e:
    raise IOError(f"An error occurred while loading the file: {e}")

In [4]:
data = pd.read_table("Hate Speech.tsv", sep= "\t")
print("Dataset Preview:")
data.head()

Dataset Preview:


,id,label,tweet
0,1,0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,2,0,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,5,0,factsguide: society now #motivation


### Data splitting

It is a good practice to split the data before EDA helps maintain the integrity of the machine learning process, prevents data leakage, simulates real-world scenarios more accurately, and ensures reliable model performance evaluation on unseen data.

In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"\nTraining Set Size: {train_df.shape}")
print(f"Testing Set Size: {test_df.shape}")


Training Set Size: (25228, 3)
Testing Set Size: (6307, 3)


### EDA on training data

- check NaNs

In [6]:
print("\nMissing Values per Column:")
print(train_df.isnull().sum())


Missing Values per Column:
id       0
label    0
tweet    0
dtype: int64


- check duplicates

In [7]:
print("\nNumber of Duplicate Rows in Training Set:")
print(train_df.duplicated().sum())


Number of Duplicate Rows in Training Set:
0


- show a representative sample of data texts to find out required preprocessing steps

In [8]:
print("\nSample Tweets:")
print(train_df['tweet'].sample(5, random_state=42))



Sample Tweets:
13979                       you make me happy! ð #me #girl #myself #and #i #snapchat #filter #nofilter #smile  â¦
8767     #like   bull up: you will dominate your bull and you will direct it whatever you want it to do. when you sta
14868               just two days away for my vintage flea market.   #fashion #vintage #fleamarket #hackneyfleamarket
17750                  best time ever!!!â¤ðððð #saturday   days #pride #columbus #ohio #fun #funtimesâ¦
23726                                                  @user what ever #decision we make it has to make us  .  #quote
Name: tweet, dtype: object


- check dataset balancing

In [9]:
print("\nClass Distribution in Training Set:")
print(train_df['label'].value_counts(normalize=True))


Class Distribution in Training Set:
label
0    0.929404
1    0.070596
Name: proportion, dtype: float64


In [10]:
train_minority = train_df[train_df['label'] == 1]
train_majority = train_df[train_df['label'] == 0]
train_minority_oversampled = resample(train_minority,
                                      replace=True,
                                      n_samples=len(train_majority),
                                      random_state=42)

train_df = pd.concat([train_majority, train_minority_oversampled])

print("Class Distribution in Training Set After Oversampling:")
print(train_df['label'].value_counts(normalize=True))

Class Distribution in Training Set After Oversampling:
label
0    0.5
1    0.5
Name: proportion, dtype: float64


- Cleaning and Preprocessing are:
    - 1
    - 2
    - 3
    - ... etc.

## Sentiment Analysis Function

In [11]:
vader = SentimentIntensityAnalyzer()

def add_sentiment_scores(df):
    df['sentiment'] = df['text'].apply(lambda x: vader.polarity_scores(x)['compound'])
    return df

### Cleaning and Preprocessing

#### Use custom scikit-learn Transformers

Using custom transformers in scikit-learn provides flexibility, reusability, and control over the data transformation process, allowing you to seamlessly integrate with scikit-learn's pipelines, enabling you to combine multiple preprocessing steps and modeling into a single workflow. This makes your code more modular, readable, and easier to maintain.

##### link: https://www.andrewvillazon.com/custom-scikit-learn-transformers/

#### Example usage:

In [12]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.tfidf_vectorizer = TfidfVectorizer(max_features=self.max_features)

    def fit(self, X, y=None):
        X_cleaned = X.apply(self.clean_text)
        self.tfidf_vectorizer.fit(X_cleaned)
        return self

    def transform(self, X, y=None):
        X_cleaned = X.apply(self.clean_text)
        X_tfidf = self.tfidf_vectorizer.transform(X_cleaned)
        return X_tfidf

    def clean_text(self, text):
        twitter_stopwords = ["rt", "via"]
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'#\w+', '', text)
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^\x00-\x7F]+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = text.lower()
        words = word_tokenize(text)
        words = [WordNetLemmatizer().lemmatize(word) for word in words if word not in stopwords.words('english') + twitter_stopwords]
        return ' '.join(words)

    def get_params(self, deep=True):
        return {"max_features": self.max_features}
# Example usage of the TextPreprocessor custom transformer
text_preprocessor = TextPreprocessor(max_features=1000)
X_train_tfidf_custom = text_preprocessor.fit_transform(train_df['tweet'])
X_test_tfidf_custom = text_preprocessor.transform(test_df['tweet'])

print(f"Custom Transformer TF-IDF Train Shape: {X_train_tfidf_custom.shape}")
print(f"Custom Transformer TF-IDF Test Shape: {X_test_tfidf_custom.shape}")

Custom Transformer TF-IDF Train Shape: (46894, 1000)
Custom Transformer TF-IDF Test Shape: (6307, 1000)


**You  are doing Great so far!**

In [13]:
df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,2,0,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,5,0,factsguide: society now #motivation


### Modelling

#### Extra: use scikit-learn pipline

##### link: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

Using pipelines in scikit-learn promotes better code organization, reproducibility, and efficiency in machine learning workflows.

#### Example usage:

In [14]:
pipeline = Pipeline([
    ('text_preprocessing', TextPreprocessor(max_features=1000)),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

In [15]:
pipeline.fit(train_df['tweet'], train_df['label'])
y_pred = pipeline.predict(test_df['tweet'])


#### Evaluation

**Evaluation metric:**
macro f1 score

Macro F1 score is a useful metric in scenarios where you want to evaluate the overall performance of a multi-class classification model, **particularly when the classes are imbalanced**

![Calculation](https://assets-global.website-files.com/5d7b77b063a9066d83e1209c/639c3d934e82c1195cdf3c60_macro-f1.webp)

In [16]:
f1 = f1_score(test_df['label'], y_pred, average='macro')
print(f"Macro F1 Score: {f1}")


Macro F1 Score: 0.6643428168807823


In [17]:
print("\nClassification Report:")
print(classification_report(test_df['label'], y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.87      0.92      5875
           1       0.29      0.70      0.41       432

    accuracy                           0.86      6307
   macro avg       0.63      0.79      0.66      6307
weighted avg       0.93      0.86      0.89      6307



#Enhancement

- Using different text representation or modeling techniques
- Hyperparameter tuning

## Grid search

In [20]:
param_grid = {
    'textpreprocessor__max_features': [500, 1000, 2000],
    'classifier__C': [0.1, 1, 10]
}


grid_search = GridSearchCV(
    estimator=Pipeline([
        ('textpreprocessor', TextPreprocessor()),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
    ]),
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=1
)

grid_search.fit(train_df['tweet'], train_df['label'])

print("\nBest Hyperparameters from GridSearchCV:")
print(grid_search.best_params_)



Best Hyperparameters from GridSearchCV:
{'classifier__C': 10, 'textpreprocessor__max_features': 500}


In [21]:
y_pred_best = grid_search.best_estimator_.predict(test_df['tweet'])

In [22]:
f1_best = f1_score(test_df['label'], y_pred_best, average='macro')
print(f"Macro F1 Score (Best Model): {f1_best}")

Macro F1 Score (Best Model): 0.660577649389795


## Ensamble method

In [24]:
ensemble_pipeline = Pipeline([
    ('text_preprocessing', TextPreprocessor(max_features=1000)),
    ('classifier', VotingClassifier(estimators=[
        ('lr', LogisticRegression()),
        ('nb', MultinomialNB()),
        ('svc', SVC(probability=True))
    ], voting='soft'))
])

ensemble_pipeline.fit(train_df['tweet'], train_df['label'])
y_pred_ensemble = ensemble_pipeline.predict(test_df['tweet'])
ensemble_f1 = f1_score(test_df['label'], y_pred_ensemble, average='macro')
print(f"Macro F1 Score (Ensemble): {ensemble_f1}")

Macro F1 Score (Ensemble): 0.7296133301106786


In [28]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    sentiment = analyzer.polarity_scores(text)
    return sentiment['compound']

# Apply sentiment score to training and test datasets
train_df['sentiment_score'] = train_df['tweet'].apply(get_sentiment_score)
test_df['sentiment_score'] = test_df['tweet'].apply(get_sentiment_score)

# For tokenization and vectorization (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Vectorize the text columns of train and test sets
X_train_tfidf = tfidf.fit_transform(train_df['tweet']).toarray()
X_test_tfidf = tfidf.transform(test_df['tweet']).toarray()

# Get sentiment scores as separate features
X_train_sentiment = train_df['sentiment_score'].values.reshape(-1, 1)
X_test_sentiment = test_df['sentiment_score'].values.reshape(-1, 1)

# Combine TF-IDF features and sentiment scores
X_train_combined = np.hstack([X_train_tfidf, X_train_sentiment])
X_test_combined = np.hstack([X_test_tfidf, X_test_sentiment])

# Encode labels (if necessary)
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])

# Check the shapes of the data
print(f"X_train_combined shape: {X_train_combined.shape}")
print(f"X_test_combined shape: {X_test_combined.shape}")

X_train_combined shape: (46894, 5001)
X_test_combined shape: (6307, 5001)


In [ ]:
def create_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model_for_grid = KerasClassifier(model=create_model, input_dim=X_train_combined.shape[1], epochs=10, batch_size=32, verbose=0)

param_grid = {
    'epochs': [5, 10],
    'batch_size': [32, 64]
}

grid = GridSearchCV(estimator=model_for_grid, param_grid=param_grid, cv= 3, n_jobs= 1, verbose=2)
grid_result = grid.fit(X_train_combined, y_train)

print("Best Parameters:", grid_result.best_params_)
print("Best Accuracy:", grid_result.best_score_)

y_pred = grid.predict(X_test_combined)

f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1 Score: {f1}")

Fitting 2 folds for each of 2 candidates, totalling 4 fits


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[CV] END ......................................batch_size=32; total time= 1.6min


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[CV] END ......................................batch_size=32; total time= 1.5min


...

[CV] END ......................................batch_size=64; total time=  57.7s


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[CV] END ......................................batch_size=64; total time=  53.7s


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Parameters: {'batch_size': 64}
Best Accuracy: 0.9887832132042479
Macro F1 Score: 0.8289754803182197


# Conclusion and final results


#conclusion 

At first I tried simple linear regression provided with adding weights to blance dataset giving Macro F1 Score: 0.6643428168807823

then I tried using Grid search with the same model and it gave me Macro F1 Score : 0.660577649389795 with Best Hyperparameters:
{'classifier__C': 10, 'textpreprocessor__max_features': 500}
which is considered not different from the first finding 

Third I tried using ensamble method which give a slight improvement with Macro F1 Score : 0.7296133301106786 with a voting clssifier combining 3 algorithms together 

In the last step I tried using neural network model after applying Vader sentiment score on the dataset with Grid search technique it gave much better Macro F1 Score: 0.8289754803182197




#Steps Summary



Data Loading and Preprocessing

Load and verify the Twitter dataset, perform an initial split for train/test to prevent data leakage, check class distribution and balance using resampling.

Text Cleaning and Vectorization

Clean the text using regex, tokenization, and lemmatization to remove unnecessary symbols and words. Your TextPreprocessor custom transformer simplifies this while enabling TF-IDF vectorization.

Sentiment Analysis and Feature Engineering

Add sentiment scores using VADER as an additional feature, which can help distinguish sentiment patterns in hate speech. Combined TF-IDF and sentiment scores create a richer representation for the model.

Model Selection and Evaluation

Experimented with both traditional models and deep learning . Evaluated with Macro F1 score due to the dataset imbalance.

Optimization

GridSearchCV was applied to optimize hyperparameters. You also tested an ensemble (Voting Classifier) to leverage multiple models, enhancing performance robustness.
Deep Learning Approach

Developed a feedforward neural network with dense layers for a neural-based approach, combining both dense layers and dropout for regularization. Further optimization with GridSearchCV helped tune this model.

#### Done!